In [0]:
%python
catalog_olist = dbutils.widgets.get('catalog')
schema_olist = dbutils.widgets.get('schema')
output_table = dbutils.widgets.get('output_table')
sk_table_olist = dbutils.widgets.get('sk_table')

In [0]:
%python
import sys 
import os

sys.path.append(os.path.abspath('..'))

from utils.utils_merge_into_tables import upsert_data

In [0]:
CREATE OR REPLACE TEMPORARY VIEW dim_tempo AS
WITH calendarDates AS (
  SELECT
    EXPLODE(array_dates) as calendar_date
  FROM(
      SELECT
        SEQUENCE(
          make_date(2016, 01, 01),
          make_date(2020, 12, 31),
          INTERVAL 1 DAY
        ) AS array_dates
    )
)
SELECT
  CAST(10000 * year(calendar_date) + 100 * month(calendar_date) + day(calendar_date) AS INT) AS SK_TEMPO,
  to_date(calendar_date) AS DATE_ACTUAL,
  year(calendar_date) AS CALENDAR_YEAR,
  quarter(calendar_date) AS CALENDAR_QUARTER, 
  month(calendar_date) AS MONTH_NUMBER,
  date_format(calendar_date, 'MMM') AS MONTH_NAME,
  day(calendar_date) AS DAY_OF_MONTH,
  dayofweek(calendar_date) AS DAY_OF_WEEK,
  CASE 
    WHEN dayofweek(calendar_date) IN (1, 7) THEN True 
    ELSE False 
  END AS IS_WEEKEND
FROM calendarDates;

## Merge Table

In [0]:
%run ../setup/00_aws_connection

In [0]:
%python
df_dim_tempo = spark.table('dim_tempo')
full_table_name = f'{catalog_olist}.{schema_olist}.{output_table}' 

upsert_data(df_dim_tempo, full_table_name,sk_table_olist,name_bucket,layer='gold')